In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('D:/my_ML/diploma_polytech/data/raw/vehicle_ins_data_1.csv', sep = ";",index_col= False)

# Создание целевой переменной: 1 если N_claims_year > 1, иначе 0
data['claim_prob'] = (data['N_claims_year'] > 1).astype(int)

# Предобработка данных
# Удаление ненужных столбцов, но НЕ УДАЛЯЕМ Date_start_contract
cols_to_drop = ['ID','Date_last_renewal', 'Date_next_renewal', 
                'Date_lapse', 'N_claims_year', 'Cost_claims_year', 'N_claims_history', 'Value_vehicle', 'Payment', 
                'R_Claims_history', 'Max_products', 'Cylinder_capacity', 'N_doors', 'Type_fuel', 'Length', 'Weight']
data = data.drop(columns=cols_to_drop)

# Преобразуем Date_start_contract в datetime ДО любой обработки
data['Date_start_contract'] = pd.to_datetime(data['Date_start_contract'], errors='coerce')#pd.to_datetime(data['Date_start_contract'], errors='coerce')

In [ ]:
# 1. Сначала обработать ВСЕ датовые колонки, если они существуют
date_columns = ['Date_birth', 'Date_driving_licence']

# Создаем список существующих датовых колонок
existing_date_columns = [col for col in date_columns if col in data.columns]

# Проверим, какие датовые колонки нашлись
print("Существующие датовые колонки для обработки:", existing_date_columns)

if not existing_date_columns:
    print("Предупреждение: Датовые колонки не найдены в DataFrame.")
else:
    for col in existing_date_columns:
        data[col] = pd.to_datetime(data[col], format='%d/%m/%Y', errors='coerce')

    # 2. Извлечь числовые признаки из существующих дат
    reference_date = pd.to_datetime('2019-12-31')
    
    if 'Date_birth' in data.columns:
        data['Age'] = (reference_date - data['Date_birth']).dt.days // 365
    
    if 'Date_driving_licence' in data.columns:
        data['Driving_experience'] = (reference_date - data['Date_driving_licence']).dt.days // 365

    

# 2. Извлечь числовые признаки из дат
reference_date = pd.to_datetime('2019-12-31')
data['Age'] = (reference_date - data['Date_birth']).dt.days // 365
data['Driving_experience'] = (reference_date - data['Date_driving_licence']).dt.days // 365

# 3. Удалить исходные датовые колонки
data = data.drop(columns=date_columns)
data = data.drop(columns=existing_date_columns, errors='ignore')
# 4. Обработать категориальные переменные
# categorical_cols = ['Type_fuel']
# label_encoders = {}
# for col in categorical_cols:
#     le = LabelEncoder()
#     data[col] = le.fit_transform(data[col].astype(str))
#     label_encoders[col] = le

# 5. Убедиться, что все данные числовые
data = data.apply(pd.to_numeric, errors='coerce')

# 6. Обработка пропущенных значений
imputer = SimpleImputer(strategy='median')
data_imputed = imputer.fit_transform(data)
data = pd.DataFrame(data_imputed, columns=data.columns)